# llama.cpp benchmark result aggregation

This notebook reads the output directory produced by `run_llamacpp_experiments.py` / `bench_llamacpp_cli_separate.py` and creates CSV exports with mean ± standard deviation for TTFT, TPOT, throughput, and peak RSS memory.

Set `ROOT_DIR` in the first code cell to the root result directory, e.g. `results/experiments-full`.

## Outputs

The notebook writes these files into `ROOT_DIR / "analysis_exports"`:

1. `all_runs_with_prompt_level_mean_std.csv`  
   One row per measured trial/run, with repeated-run mean/std columns attached for the same configuration and prompt.

2. `configuration_category_and_overall_mean_std.csv`  
   One row per configuration × prompt category (`short`, `medium`, `long`) plus one `overall` row per configuration.

3. `raw_measured_runs.csv`  
   A plain measured-run table, useful for auditing.

Metric note: the benchmark currently measures visible stdout byte events, not true model tokens. Therefore TPOT and throughput are event-based unless the benchmark script is changed to expose token-level timings.

In [1]:
from pathlib import Path

# Change this to your benchmark result directory.
# Example from run_bench.sh: results/experiments-full
ROOT_DIR = Path("data/experiments")

OUTPUT_DIR = ROOT_DIR / "analysis_exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR = {ROOT_DIR.resolve()}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")

ROOT_DIR = C:\Users\morit\OneDrive - TU Wien\Dokumente\Informatik\SS26\BDA\Inference\data\experiments
OUTPUT_DIR = C:\Users\morit\OneDrive - TU Wien\Dokumente\Informatik\SS26\BDA\Inference\data\experiments\analysis_exports


In [2]:
import json
import re
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

METRIC_SOURCE_COLUMNS = {
    "ttft_s": "TTFT (s)",
    "tpot_s": "TPOT (s/event)",
    "throughput_events_per_s": "Throughput (events/s)",
    "peak_rss_mb": "Peak RSS (MB)",
}

NUMERIC_METRICS = list(METRIC_SOURCE_COLUMNS.keys())

In [ ]:
def _safe_read_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {}
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


def _parse_int_from_name(value: str) -> Optional[int]:
    match = re.search(r"(-?\d+)", value)
    return int(match.group(1)) if match else None


def _as_bool_not_warmup(series: pd.Series) -> pd.Series:
    """Return True for measured rows and False for warmup rows."""
    if series.dtype == bool:
        return ~series
    text = series.astype(str).str.strip().str.lower()
    return ~text.isin({"true", "1", "yes", "y"})


def infer_configuration(summary_csv: Path, root_dir: Path) -> Dict[str, Any]:
    """Infer experiment group and configuration value from a summary.csv path."""
    rel = summary_csv.relative_to(root_dir)
    parts = rel.parts[:-1]  # drop summary.csv

    info: Dict[str, Any] = {
        "configuration_group": "unknown",
        "configuration_name": "unknown",
        "configuration_value": None,
        "configuration_label": "unknown",
        "worker_id": None,
        "condition_dir": str(summary_csv.parent),
        "summary_csv": str(summary_csv),
    }

    if len(parts) >= 2 and parts[0] == "threads" and parts[1].startswith("threads_"):
        value = _parse_int_from_name(parts[1])
        info.update({
            "configuration_group": "threads",
            "configuration_name": "threads",
            "configuration_value": value,
            "configuration_label": f"threads={value}",
        })
    elif len(parts) >= 2 and parts[0] == "decode" and parts[1].startswith("decode_"):
        value = _parse_int_from_name(parts[1])
        info.update({
            "configuration_group": "decode_length",
            "configuration_name": "n_predict",
            "configuration_value": value,
            "configuration_label": f"n_predict={value}",
        })
    elif len(parts) >= 2 and parts[0] == "context" and parts[1].startswith("context_"):
        value = _parse_int_from_name(parts[1])
        info.update({
            "configuration_group": "context_length",
            "configuration_name": "target_context_tokens_approx",
            "configuration_value": value,
            "configuration_label": f"target_context_tokens≈{value}",
        })
    elif len(parts) >= 2 and parts[0] == "concurrency" and parts[1].startswith("concurrency_"):
        value = _parse_int_from_name(parts[1])
        worker_id = None
        if len(parts) >= 3 and parts[2].startswith("worker_"):
            worker_id = _parse_int_from_name(parts[2])
        info.update({
            "configuration_group": "concurrency",
            "configuration_name": "concurrency",
            "configuration_value": value,
            "configuration_label": f"concurrency={value}",
            "worker_id": worker_id,
        })
    elif len(parts) >= 1:
        # Fallback for custom experiment folders.
        group = parts[0]
        value_name = parts[1] if len(parts) > 1 else group
        value = _parse_int_from_name(value_name)
        info.update({
            "configuration_group": group,
            "configuration_name": value_name,
            "configuration_value": value,
            "configuration_label": "/".join(parts),
        })

    return info


def load_one_summary(summary_csv: Path, root_dir: Path) -> pd.DataFrame:
    config_info = infer_configuration(summary_csv, root_dir)
    df = pd.read_csv(summary_csv)

    if "warmup" in df.columns:
        df = df[_as_bool_not_warmup(df["warmup"])].copy()

    # Standardize peak memory naming to describe implementation: sampled process RSS.
    if "memory_peak_mb" in df.columns and "peak_rss_mb" not in df.columns:
        df["peak_rss_mb"] = df["memory_peak_mb"]

    # Attach path-inferred configuration info.
    for key, value in config_info.items():
        df[key] = value

    # Attach per-run benchmark config if available.
    bench_cfg = _safe_read_json(summary_csv.parent / "benchmark_config.json")
    for key in ["model", "prompt_file", "n_predict", "runs", "warmup_runs", "seed", "threads", "ctx_size", "monitor", "rapl"]:
        df[f"benchmark_{key}"] = bench_cfg.get(key)

    # Coerce metrics and common identifiers.
    for col in NUMERIC_METRICS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            df[col] = np.nan

    if "trial" in df.columns:
        df["trial"] = pd.to_numeric(df["trial"], errors="coerce").astype("Int64")

    return df


def discover_and_load(root_dir: Path) -> pd.DataFrame:
    summary_files = sorted(root_dir.rglob("summary.csv"))
    summary_files = [p for p in summary_files if "analysis_exports" not in p.parts]
    if not summary_files:
        raise FileNotFoundError(f"No summary.csv files found under {root_dir.resolve()}")

    frames = []
    for path in summary_files:
        try:
            frames.append(load_one_summary(path, root_dir))
        except Exception as exc:
            print(f"Skipping {path}: {exc}")

    if not frames:
        raise RuntimeError("Found summary.csv files, but none could be loaded successfully.")

    out = pd.concat(frames, ignore_index=True, sort=False)
    out = out.drop(columns=['generated_output'])
    # Keep a stable, readable column order while retaining extra columns at the end.
    preferred = [
        "configuration_group", "configuration_name", "configuration_value", "configuration_label", "worker_id",
        "prompt_id", "category", "mandatory", "trial", "warmup",
        "ttft_s", "tpot_s", "throughput_events_per_s", "peak_rss_mb",
        "output_events_count", "total_time_s", "energy_j",
        "benchmark_threads", "benchmark_ctx_size", "benchmark_n_predict", "benchmark_runs", "benchmark_warmup_runs",
        "condition_dir", "summary_csv", "raw_json", "resource_csv",
    ]
    ordered = [c for c in preferred if c in out.columns] + [c for c in out.columns if c not in preferred]
    return out[ordered]


raw_runs = discover_and_load(ROOT_DIR)
print(f"Loaded measured rows: {len(raw_runs):,}")
print(f"Configurations: {raw_runs['configuration_label'].nunique():,}")
display(raw_runs.head())

Loaded measured rows: 180
Configurations: 18


,configuration_group,configuration_name,configuration_value,configuration_label,worker_id,prompt_id,category,mandatory,trial,warmup,ttft_s,tpot_s,throughput_events_per_s,peak_rss_mb,output_events_count,total_time_s,energy_j,benchmark_threads,benchmark_ctx_size,benchmark_n_predict,benchmark_runs,benchmark_warmup_runs,condition_dir,summary_csv,raw_json,resource_csv,memory_peak_mb,generated_output,benchmark_model,benchmark_prompt_file,benchmark_seed,benchmark_monitor,benchmark_rapl
0,concurrency,concurrency,1,concurrency=1,0,mandatory_short_capital_france,short,True,0,False,1.912671,0.000436,2289.583200,348.0625,968,2.335455,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_1\wor...,data\experiments\concurrency\concurrency_1\wor...,results/experiments/concurrency/concurrency_1/...,results/experiments/concurrency/concurrency_1/...,348.0625,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False
1,concurrency,concurrency,1,concurrency=1,0,mandatory_medium_ml_vs_dl,medium,True,0,False,1.960854,0.003580,279.472599,354.0000,1727,8.140351,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_1\wor...,data\experiments\concurrency\concurrency_1\wor...,results/experiments/concurrency/concurrency_1/...,results/experiments/concurrency/concurrency_1/...,354.0000,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False
2,concurrency,concurrency,1,concurrency=1,0,mandatory_long_transformer_cpu_quantization,long,True,0,False,3.789583,0.003158,316.689244,359.9375,2097,10.411216,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_1\wor...,data\experiments\concurrency\concurrency_1\wor...,results/experiments/concurrency/concurrency_1/...,results/experiments/concurrency/concurrency_1/...,359.9375,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False
3,concurrency,concurrency,16,concurrency=16,0,mandatory_short_capital_france,short,True,0,False,126.226996,0.006462,154.897914,347.0625,966,132.463362,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_16\wo...,data\experiments\concurrency\concurrency_16\wo...,results/experiments/concurrency/concurrency_16...,results/experiments/concurrency/concurrency_16...,347.0625,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False
4,concurrency,concurrency,16,concurrency=16,0,mandatory_medium_ml_vs_dl,medium,True,0,False,102.617702,0.062076,16.118668,349.0000,1725,209.636467,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_16\wo...,data\experiments\concurrency\concurrency_16\wo...,results/experiments/concurrency/concurrency_16...,results/experiments/concurrency/concurrency_16...,349.0000,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False


In [4]:
# Save a raw measured-run table for auditing.
raw_runs_path = OUTPUT_DIR / "raw_measured_runs.csv"
raw_runs.to_csv(raw_runs_path, index=False)
print(f"Wrote {raw_runs_path}")

Wrote data\experiments\analysis_exports\raw_measured_runs.csv


In [5]:
def std_sample_zero_for_single(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) <= 1:
        return 0.0 if len(x) == 1 else np.nan
    return float(x.std(ddof=1))


def mean_value(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.mean()) if len(x) else np.nan


def format_mean_std(mean: Any, std: Any, decimals: int = 6) -> str:
    if pd.isna(mean):
        return ""
    std = 0.0 if pd.isna(std) else std
    return f"{mean:.{decimals}f} ± {std:.{decimals}f}"


def add_metric_mean_std_columns(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Attach mean/std for each metric to every row, computed over group_cols."""
    out = df.copy()
    grouped = out.groupby(group_cols, dropna=False)
    for metric in NUMERIC_METRICS:
        out[f"{metric}_mean"] = grouped[metric].transform(lambda s: mean_value(s))
        out[f"{metric}_std"] = grouped[metric].transform(lambda s: std_sample_zero_for_single(s))
        out[f"{metric}_mean_pm_std"] = [
            format_mean_std(m, s) for m, s in zip(out[f"{metric}_mean"], out[f"{metric}_std"])
        ]
    return out

# Same prompt/configuration = repeated trials for that prompt under that configuration.
# For concurrency, workers remain individual observations but the mean/std is computed across all worker trials for the same config+prompt.
prompt_level_group_cols = [
    "configuration_group", "configuration_name", "configuration_value", "configuration_label",
    "prompt_id", "category",
]

all_runs_with_stats = add_metric_mean_std_columns(raw_runs, prompt_level_group_cols)

# Helpful observation counts for repeated-run context.
counts = (
    raw_runs.groupby(prompt_level_group_cols, dropna=False)
    .agg(n_observations=("prompt_id", "size"), n_workers=("worker_id", lambda s: s.dropna().nunique()))
    .reset_index()
)
all_runs_with_stats = all_runs_with_stats.merge(counts, on=prompt_level_group_cols, how="left")

all_runs_path = OUTPUT_DIR / "all_runs_with_prompt_level_mean_std.csv"
all_runs_with_stats.to_csv(all_runs_path, index=False)
print(f"Wrote {all_runs_path}")
display(all_runs_with_stats.head())

Wrote data\experiments\analysis_exports\all_runs_with_prompt_level_mean_std.csv


,configuration_group,configuration_name,configuration_value,configuration_label,worker_id,prompt_id,category,mandatory,trial,warmup,ttft_s,tpot_s,throughput_events_per_s,peak_rss_mb,output_events_count,total_time_s,energy_j,benchmark_threads,benchmark_ctx_size,benchmark_n_predict,benchmark_runs,benchmark_warmup_runs,condition_dir,summary_csv,raw_json,resource_csv,memory_peak_mb,generated_output,benchmark_model,benchmark_prompt_file,benchmark_seed,benchmark_monitor,benchmark_rapl,ttft_s_mean,ttft_s_std,ttft_s_mean_pm_std,tpot_s_mean,tpot_s_std,tpot_s_mean_pm_std,throughput_events_per_s_mean,throughput_events_per_s_std,throughput_events_per_s_mean_pm_std,peak_rss_mb_mean,peak_rss_mb_std,peak_rss_mb_mean_pm_std,n_observations,n_workers
0,concurrency,concurrency,1,concurrency=1,0,mandatory_short_capital_france,short,True,0,False,1.912671,0.000436,2289.583200,348.0625,968,2.335455,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_1\wor...,data\experiments\concurrency\concurrency_1\wor...,results/experiments/concurrency/concurrency_1/...,results/experiments/concurrency/concurrency_1/...,348.0625,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False,1.912671,0.000000,1.912671 ± 0.000000,0.000436,0.000000,0.000436 ± 0.000000,2289.583200,0.000000,2289.583200 ± 0.000000,348.062500,0.000000,348.062500 ± 0.000000,1,1
1,concurrency,concurrency,1,concurrency=1,0,mandatory_medium_ml_vs_dl,medium,True,0,False,1.960854,0.003580,279.472599,354.0000,1727,8.140351,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_1\wor...,data\experiments\concurrency\concurrency_1\wor...,results/experiments/concurrency/concurrency_1/...,results/experiments/concurrency/concurrency_1/...,354.0000,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False,1.960854,0.000000,1.960854 ± 0.000000,0.003580,0.000000,0.003580 ± 0.000000,279.472599,0.000000,279.472599 ± 0.000000,354.000000,0.000000,354.000000 ± 0.000000,1,1
2,concurrency,concurrency,1,concurrency=1,0,mandatory_long_transformer_cpu_quantization,long,True,0,False,3.789583,0.003158,316.689244,359.9375,2097,10.411216,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_1\wor...,data\experiments\concurrency\concurrency_1\wor...,results/experiments/concurrency/concurrency_1/...,results/experiments/concurrency/concurrency_1/...,359.9375,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False,3.789583,0.000000,3.789583 ± 0.000000,0.003158,0.000000,0.003158 ± 0.000000,316.689244,0.000000,316.689244 ± 0.000000,359.937500,0.000000,359.937500 ± 0.000000,1,1
3,concurrency,concurrency,16,concurrency=16,0,mandatory_short_capital_france,short,True,0,False,126.226996,0.006462,154.897914,347.0625,966,132.463362,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_16\wo...,data\experiments\concurrency\concurrency_16\wo...,results/experiments/concurrency/concurrency_16...,results/experiments/concurrency/concurrency_16...,347.0625,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct-Q4_K_M.gguf,results/experiments/workloads/sweep_workload.json,42,True,False,124.359293,3.950339,124.359293 ± 3.950339,0.006461,0.000147,0.006461 ± 0.000147,154.831611,3.610042,154.831611 ± 3.610042,348.632812,1.067970,348.632812 ± 1.067970,16,16
4,concurrency,concurrency,16,concurrency=16,0,mandatory_medium_ml_vs_dl,medium,True,0,False,102.617702,0.062076,16.118668,349.0000,1725,209.636467,NaN,4,None,128,1,1,data\experiments\concurrency\concurrency_16\wo...,data\experiments\concurrency\concurrency_16\wo...,results/experiments/concurrency/concurrency_16...,results/experiments/concurrency/concurrency_16...,349.0000,\nLoading model... \n\n\n▄▄ ▄▄\n██ ██\n██ ██ ...,../models/SmolLM2-135M-Instruct

In [6]:
def summarize_by(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    base = (
        df.groupby(group_cols, dropna=False)
        .agg(
            n_observations=("prompt_id", "size"),
            n_prompts=("prompt_id", "nunique"),
            n_workers=("worker_id", lambda s: s.dropna().nunique()),
        )
        .reset_index()
    )

    metric_parts = []
    for metric in NUMERIC_METRICS:
        part = (
            df.groupby(group_cols, dropna=False)[metric]
            .agg(**{f"{metric}_mean": mean_value, f"{metric}_std": std_sample_zero_for_single})
            .reset_index()
        )
        part[f"{metric}_mean_pm_std"] = [
            format_mean_std(m, s) for m, s in zip(part[f"{metric}_mean"], part[f"{metric}_std"])
        ]
        metric_parts.append(part)

    out = base
    for part in metric_parts:
        out = out.merge(part, on=group_cols, how="left")
    return out

config_cols = ["configuration_group", "configuration_name", "configuration_value", "configuration_label"]

by_category = summarize_by(raw_runs, config_cols + ["category"])

overall_input = raw_runs.copy()
overall_input["category"] = "overall"
overall = summarize_by(overall_input, config_cols + ["category"])

clustered = pd.concat([by_category, overall], ignore_index=True, sort=False)

# Sort categories in a useful order.
cat_order = {"short": 0, "medium": 1, "long": 2, "overall": 3}
clustered["_cat_order"] = clustered["category"].map(cat_order).fillna(99)
clustered = clustered.sort_values(
    ["configuration_group", "configuration_value", "_cat_order", "category"],
    kind="stable"
).drop(columns=["_cat_order"])

clustered_path = OUTPUT_DIR / "configuration_category_and_overall_mean_std.csv"
clustered.to_csv(clustered_path, index=False)
print(f"Wrote {clustered_path}")
display(clustered.head(20))

Wrote data\experiments\analysis_exports\configuration_category_and_overall_mean_std.csv


,configuration_group,configuration_name,configuration_value,configuration_label,category,n_observations,n_prompts,n_workers,ttft_s_mean,ttft_s_std,ttft_s_mean_pm_std,tpot_s_mean,tpot_s_std,tpot_s_mean_pm_std,throughput_events_per_s_mean,throughput_events_per_s_std,throughput_events_per_s_mean_pm_std,peak_rss_mb_mean,peak_rss_mb_std,peak_rss_mb_mean_pm_std
2,concurrency,concurrency,1,concurrency=1,short,1,1,1,1.912671,0.000000,1.912671 ± 0.000000,0.000436,0.000000,0.000436 ± 0.000000,2289.583200,0.000000,2289.583200 ± 0.000000,348.062500,0.000000,348.062500 ± 0.000000
1,concurrency,concurrency,1,concurrency=1,medium,1,1,1,1.960854,0.000000,1.960854 ± 0.000000,0.003580,0.000000,0.003580 ± 0.000000,279.472599,0.000000,279.472599 ± 0.000000,354.000000,0.000000,354.000000 ± 0.000000
0,concurrency,concurrency,1,concurrency=1,long,1,1,1,3.789583,0.000000,3.789583 ± 0.000000,0.003158,0.000000,0.003158 ± 0.000000,316.689244,0.000000,316.689244 ± 0.000000,359.937500,0.000000,359.937500 ± 0.000000
44,concurrency,concurrency,1,concurrency=1,overall,3,3,1,2.554369,1.069997,2.554369 ± 1.069997,0.002392,0.001706,0.002392 ± 0.001706,961.915014,1149.944946,961.915014 ± 1149.944946,354.000000,5.937500,354.000000 ± 5.937500
5,concurrency,concurrency,2,concurrency=2,short,2,1,2,3.683269,0.064362,3.683269 ± 0.064362,0.000684,0.000021,0.000684 ± 0.000021,1463.363178,44.247292,1463.363178 ± 44.247292,349.625000,0.000000,349.625000 ± 0.000000
4,concurrency,concurrency,2,concurrency=2,medium,2,1,2,3.440877,0.087309,3.440877 ± 0.087309,0.007039,0.000032,0.007039 ± 0.000032,142.134460,0.643955,142.134460 ± 0.643955,354.312500,0.088388,354.312500 ± 0.088388
3,concurrency,concurrency,2,concurrency=2,long,2,1,2,6.687401,0.299617,6.687401 ± 0.299617,0.006163,0.000102,0.006163 ± 0.000102,162.356950,2.675700,162.356950 ± 2.675700,360.000000,0.000000,360.000000 ± 0.000000
45,concurrency,concurrency,2,concurrency=2,overall,6,3,2,4.603849,1.623814,4.603849 ± 1.623814,0.004629,0.003081,0.004629 ± 0.003081,589.284863,677.408747,589.284863 ± 677.408747,354.645833,4.647188,354.645833 ± 4.647188
8,concurrency,concurrency,4,concurrency=4,short,4,1,4,6.969892,0.091201,6.969892 ± 0.091201,0.001596,0.000033,0.001596 ± 0.000033,627.105113,12.828983,627.105113 ± 12.828983,348.921875,1.098502,348.921875 ± 1.098502
7,concurrency,concurrency,4,concurrency=4,medium,4,1,4,6.864995,0.174012,6.864995 ± 0.174012,0.014622,0.000074,0.014622 ± 0.000074,68.429586,0.345409,68.429586 ± 0.345409,351.953125,2.595256,351.953125 ± 2.595256


In [7]:
# Optional compact check: show only the formatted mean ± std columns.
formatted_cols = [
    "configuration_label", "category", "n_observations", "n_prompts", "n_workers",
    "ttft_s_mean_pm_std", "tpot_s_mean_pm_std",
    "throughput_events_per_s_mean_pm_std", "peak_rss_mb_mean_pm_std",
]
existing = [c for c in formatted_cols if c in clustered.columns]
display(clustered[existing])

,configuration_label,category,n_observations,n_prompts,n_workers,ttft_s_mean_pm_std,tpot_s_mean_pm_std,throughput_events_per_s_mean_pm_std,peak_rss_mb_mean_pm_std
2,concurrency=1,short,1,1,1,1.912671 ± 0.000000,0.000436 ± 0.000000,2289.583200 ± 0.000000,348.062500 ± 0.000000
1,concurrency=1,medium,1,1,1,1.960854 ± 0.000000,0.003580 ± 0.000000,279.472599 ± 0.000000,354.000000 ± 0.000000
0,concurrency=1,long,1,1,1,3.789583 ± 0.000000,0.003158 ± 0.000000,316.689244 ± 0.000000,359.937500 ± 0.000000
44,concurrency=1,overall,3,3,1,2.554369 ± 1.069997,0.002392 ± 0.001706,961.915014 ± 1149.944946,354.000000 ± 5.937500
5,concurrency=2,short,2,1,2,3.683269 ± 0.064362,0.000684 ± 0.000021,1463.363178 ± 44.247292,349.625000 ± 0.000000
...,...,...,...,...,...,...,...,...,...
60,threads=32,overall,9,3,0,2.334300 ± 0.418017,0.003918 ± 0.002453,588.300967 ± 608.495534,354.479167 ± 5.166940
43,threads=48,short,3,1,0,2.176664 ± 0.002320,0.000942 ± 0.000028,1058.071254 ± 30.982747,328.020833 ± 0.072169
42,threads=48,medium,3,1,0,2.195892 ± 0.013559,0.008489 ± 0.000588,118.197989 ± 7.876386,328.500000 ± 0.165359
41,threads=48,long,3,1,0,2.949063 ± 0.001342,0.006728 ± 0.000021,148.645391 ± 0.467052,330.145833 ± 0.144338


## Notes on how the current benchmark handles custom context lengths

The context-length experiment does **not** select existing prompts of a matching length. It generates a synthetic one-prompt JSON file for each target context size. The prompt is built by repeating a fixed base sentence until it reaches an approximate word count, then adding a question. The target is approximate: the code assumes roughly `1 token ≈ 0.75 words`; it does not tokenize with the actual model tokenizer.

For each context target, the runner writes a prompt file such as `context_512.json`, with prompt id `context_512`. It assigns the category automatically: `<512 → short`, `512–1023 → medium`, and `>=1024 → long`.

The llama.cpp context window passed to `llama-cli` is set separately. For each context experiment, `ctx_size = max(target_context_tokens + n_predict + 64, args.ctx_size or 0)`, so the configured context window is at least the approximate prompt target plus output budget and a safety margin.

In [8]:
print("Done. Main CSV outputs:")
print(f"1. {all_runs_path}")
print(f"2. {clustered_path}")
print(f"Audit: {raw_runs_path}")

Done. Main CSV outputs:
1. data\experiments\analysis_exports\all_runs_with_prompt_level_mean_std.csv
2. data\experiments\analysis_exports\configuration_category_and_overall_mean_std.csv
Audit: data\experiments\analysis_exports\raw_measured_runs.csv
